# Feature Engineering

In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn 
import math

In [3]:
#importing both the files
og_df = pd.read_parquet(r"C:\Users\Niraj Mhatre\projects\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework\Data\Data_processed\master_origination_clean.parquet")
m_df = pd.read_parquet(
    r"C:\Users\Niraj Mhatre\projects\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework\Data\Data_processed\master_monthly_performance_clean.parquet")

## we split before encoding

In [4]:
train_df = og_df[og_df["origination_year"].between(2000, 2006)].copy()
val_df = og_df[og_df["origination_year"] == 2007].copy()
test_df = og_df[og_df["origination_year"].between(2008, 2010)].copy()

now a crazy idea for splitting panel data

In [5]:
print("="*15 + " SPLITTING LONGITUDINAL PANEL DATA " + "="*15)

# 1. Extract the unique, isolated loan keys from your origination dataframes
train_ids = train_df["loan_seq_no"]
val_ids   = val_df["loan_seq_no"]
test_ids  = test_df["loan_seq_no"]

# 2. Slice the master performance dataframe using these keys
m_train_df = m_df[m_df["loan_seq_no"].isin(train_ids)].copy()
m_val_df   = m_df[m_df["loan_seq_no"].isin(val_ids)].copy()
m_test_df  = m_df[m_df["loan_seq_no"].isin(test_ids)].copy()

# 3. Quick structural verification printout
print(f"✅ Master Performance Rows: {len(m_df):,}")
print(f"   -> Train Performance Panel (00-06 loans): {len(m_train_df):,} rows")
print(f"   -> Val Performance Panel   (2007 loans):   {len(m_val_df):,} rows")
print(f"   -> Test Performance Panel  (08-10 loans):  {len(m_test_df):,} rows")

# Double check that no rows were dropped or duplicated accidentally
assert len(m_train_df) + len(m_val_df) + len(m_test_df) == len(m_df), "Mismatch in total panel row allocation!"

=============== SPLITTING LONGITUDINAL PANEL DATA ===============
✅ Master Performance Rows: 33,058,391
   -> Train Performance Panel (00-06 loans): 21,032,440 rows
   -> Val Performance Panel   (2007 loans):   3,003,932 rows
   -> Test Performance Panel  (08-10 loans):  9,022,019 rows


#===================================================
#SPLIT USING SKLEARN THIS IS ANOTHER WAY TO DO IT
#====================================================
from sklearn.model_selection import PredefinedSplit

#1. Create a tracking array map initialized to -1 (defaulting to Train)
split_indices = np.full(shape=len(og_df), fill_value=-1)

#2. Assign structural flags matching sklearn's validation scheme
#In PredefinedSplit: -1 = Train, 0 = Test/Validation
#We isolate 2007 as our designated validation fold
split_indices[og_df["origination_year"] == 2007] = 0

#3. Initialize the sklearn PredefinedSplit engine
custom_cv = PredefinedSplit(test_fold=split_indices)

#4. Extract the train and validation subsets using sklearn's split generator
for train_idx, val_idx in custom_cv.split():
    # This automatically puts 2000-2006 AND 2008-2010 into train_df_raw
    train_df_raw = og_df.iloc[train_idx]
    val_df = og_df.iloc[val_idx].copy()

#5. Cleanly separate your Train and out-of-time Test sets from the raw train split
train_df = train_df_raw[train_df_raw["origination_year"] <= 2006].copy()
test_df = train_df_raw[train_df_raw["origination_year"] >= 2008].copy()

#6. Verification check
print(f"Train rows (2000-2006): {len(train_df):,}")
print(f"Val rows   (2007):      {len(val_df):,}")
print(f"Test rows  (2008-2010): {len(test_df):,}")

In [6]:
print("="*20 + " STARTING TRANSFORMATION PIPELINE " + "="*20)

# 1. MASK SENTINELS (Apply the 'between' rule to all three sets to drop 9999s to NaN)
train_df["cred_score"] = train_df["cred_score"].where(train_df["cred_score"].between(300, 850), np.nan)
val_df["cred_score"]   = val_df["cred_score"].where(val_df["cred_score"].between(300, 850), np.nan)
test_df["cred_score"]  = test_df["cred_score"].where(test_df["cred_score"].between(300, 850), np.nan)


# 3. FIT (Calculate the median using ONLY the clean training data)
train_fico_median = train_df["cred_score"].median()


# 4. TRANSFORM (Impute the saved training median into the NaNs of ALL sets)
train_df["cred_score"] = train_df["cred_score"].fillna(train_fico_median)
val_df["cred_score"]   = val_df["cred_score"].fillna(train_fico_median)
test_df["cred_score"]  = test_df["cred_score"].fillna(train_fico_median)

print(f"Missing values remaining -> Train: {train_df['cred_score'].isna().sum()} | Val: {val_df['cred_score'].isna().sum()} | Test: {test_df['cred_score'].isna().sum()}")

==================== STARTING TRANSFORMATION PIPELINE ====================
Missing values remaining -> Train: 0 | Val: 0 | Test: 0


In [7]:
all_sets = [train_df, val_df, test_df]


# =========================================================================
# STEP 1: MASK SENTINELS & OUT-OF-BOUNDS NOISE TO NaN
# =========================================================================
for df in all_sets:
    # og_dti: valid range 6% to 200% (999 or anything else becomes NaN)
    df["og_dti"] = df["og_dti"].where(df["og_dti"].between(6, 200), np.nan)
    
    # og_cltv: valid range 6% to 105% 
    df["og_cltv"] = df["og_cltv"].where(df["og_cltv"].between(6, 105), np.nan)
    
    # og_ltv: valid range 6% to 105%
    df["og_ltv"] = df["og_ltv"].where(df["og_ltv"].between(6, 105), np.nan)
    
    # mortgage_insurance_percent: valid range 1% to 55%. 
    # Note: 0 means "No MI" (valid economic status), so we keep 0 and mask 999/out-of-bounds.
    df["mortgage_insurance_percent"] = df["mortgage_insurance_percent"].where(
        (df["mortgage_insurance_percent"] == 0) | df["mortgage_insurance_percent"].between(1, 55), 
        np.nan
    )


# =========================================================================
# STEP 2: FIT & TRANSFORM (Compute median on Train, apply to all)
# =========================================================================
target_vars = ["og_dti", "og_cltv", "og_ltv", "mortgage_insurance_percent"]

print("--- IMPUTATION SUMMARY ---")
for var in target_vars:
    # Fit: Calculate the robust median strictly from the clean training set
    train_median = train_df[var].median()
    
    # Transform: Patch the missing gaps across Train, Val, and Test
    train_df[var] = train_df[var].fillna(train_median)
    val_df[var]   = val_df[var].fillna(train_median)
    test_df[var]  = test_df[var].fillna(train_median)
    
    print(f"🔹 {var:28} | Training Median: {train_median:<5.1f} | Rem. NaNs (Train/Val/Test): {train_df[var].isna().sum()}/{val_df[var].isna().sum()}/{test_df[var].isna().sum()}")

--- IMPUTATION SUMMARY ---
🔹 og_dti                       | Training Median: 33.0  | Rem. NaNs (Train/Val/Test): 0/0/0
🔹 og_cltv                      | Training Median: 75.0  | Rem. NaNs (Train/Val/Test): 0/0/0
🔹 og_ltv                       | Training Median: 75.0  | Rem. NaNs (Train/Val/Test): 0/0/0
🔹 mortgage_insurance_percent   | Training Median: 0.0   | Rem. NaNs (Train/Val/Test): 0/0/0


In [8]:
na_val = m_df.isnull().sum()
print(na_val[na_val > 0] / len(m_df))

remaining_months_to_legal_maturity    0.000009
defect_settlement_date                0.999892
zero_balance_code                     0.983764
zero_balance_effective_date           0.983764
ddlpi                                 0.998187
mi_recoveries                         0.999477
net_sale_proceeds                     0.999475
non_mi_recoveries                     0.999477
total_expenses                        0.999477
legal_costs                           0.999477
maintenance_and_preservation_costs    0.999477
taxes_and_insurance                   0.999477
miscellaneous_expenses                0.999477
actual_loss_calculation               0.999475
cumulative_modification_cost          0.999655
interest_rate_step_indicator          0.971975
eltv                                  0.903336
zero_balance_removal_upb              0.983764
delinquent_accrued_interest           0.999475
delinquency_due_to_disaster           0.998891
borrower_assistance_status_code       0.998154
current_month

In [ ]:
#imputing performance data
# imputing performance data
# compute year-wise median eltv from training panel and impute into train/val/test panels
eltv_median_by_year = m_train_df.groupby("origination_year")["eltv"].median()
overall_eltv_median = m_train_df["eltv"].median()

for df in (m_train_df, m_val_df, m_test_df):
    df["eltv"] = df["eltv"].fillna(df["origination_year"].map(eltv_median_by_year))
    df["eltv"] = df["eltv"].fillna(overall_eltv_median)

print("Remaining NaNs (train/val/test):",
      m_train_df["eltv"].isna().sum(), m_val_df["eltv"].isna().sum(), m_test_df["eltv"].isna().sum())

MemoryError: Unable to allocate 252. MiB for an array with shape (33058391, 1) and data type object